# 22DM015 Final Project — Financial PhraseBank
## Person B: Part 2 — BERT track (BERT, augmentation eval)

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).‍
**Labels:** 0 = negative, 1 = neutral, 2 = positive.‍

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453); the 32-shot set lives separately at `data/labeled_32.csv` and is loaded with `pd.read_csv` (no longer returned by `du.load_splits`).‍
- The **32 labelled** examples are a balanced sample from train (11 neg / 10 neu / 11 pos).‍
- Part 2 'unlabelled' data = train minus the 32.‍
- Evaluate headline numbers on **`test`** only; tune on **`val`**.‍ Use `eval_utils.evaluate` so we're comparable.‍
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.‍
- Part 3 (full-data SOA curve) lives in `notebooks/part3.ipynb`.‍

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule).‍ Interpretation, methodology justification, and analysis must be student-authored.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
import pandas as pd
SEED = 618
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical train/val/test for everyone
train, val, test = splits['train'], splits['val'], splits['test']
# labeled_32 is no longer part of load_splits — load the committed CSV directly.
labeled_32 = pd.read_csv('../data/labeled_32.csv')
# Part 2 'unlabelled' data = train rows not among the 32 labelled ones (computed locally
# now that du.unlabeled_pool's contract may change with labeled_32 leaving load_splits).
pool = train[~train['id'].isin(set(labeled_32['id']))].reset_index(drop=True)
for k, v in {**splits, 'labeled_32': labeled_32}.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

> **Install (run once):** `transformers`, `torch`, `accelerate` are needed here.‍ On Python 3.14 torch wheels may be missing — use a 3.11/3.12 venv.‍

## Part 2a.‍ BERT with 32 labelled examples (0.5)
Fine-tune a BERT-family model on `labeled_32`, evaluate on `test`.‍ Expect instability with 32 examples — fix seed, report val + test.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2a: fine-tune a BERT classifier on the 32 labelled examples; evaluate on the shared
# test split. Also defines the helpers every later experiment cell reuses (one shared
# training protocol -> all rows in results.csv stay comparable).
import os, logging, warnings
# Tell transformers to skip its TensorFlow/Flax/Keras probe (we use PyTorch only).
# This prevents tf_keras from being imported, which is what emits the
# "tf.losses.sparse_softmax_cross_entropy is deprecated" notice. MUST be set BEFORE
# importing transformers.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # silence TF C++ logs if it still loads
logging.getLogger("tensorflow").setLevel(logging.ERROR)
# Silence the cosmetic "Redirects are currently not supported in Windows or MacOs" NOTE
# emitted by torch.distributed.elastic.multiprocessing.redirects when Trainer is imported
# on Windows/macOS. No-op on Linux. Set BEFORE importing transformers.
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
# Silence torchao's "Skipping import of cpp extensions due to incompatible torch version"
# message (https://github.com/pytorch/ao/issues/2919). We don't use torchao kernels here,
# so the Python fallbacks are fine. Covers both the logger and warnings.warn paths.
logging.getLogger("torchao").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=r".*Skipping import of cpp extensions.*")

import pandas as pd, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, set_seed)

torch.set_num_threads(os.cpu_count() or 4)   # all trainings here are CPU-bound
MODEL = 'bert-base-uncased'      # general BERT; 'ProsusAI/finbert' reserved for the Part 3 SOA comparison
NUM_LABELS = 3
MAX_LEN = 128
AUG_CSV = '../data/augmented_32.csv'
GEN_CSV = '../data/llm_generated.csv'

tok = AutoTokenizer.from_pretrained(MODEL)


def encode(df, max_len=MAX_LEN):
    ds = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
    return ds.map(lambda b: tok(b['text'], truncation=True, padding='max_length', max_length=max_len),
                  batched=True)


def train_bert(train_df, out_dir, *, epochs=20, batch=8, max_len=MAX_LEN, log_epochs=False):
    """One shared fine-tuning protocol for every experiment in this notebook.
    Part 2 default: max_len 128 / batch 8 / 20 epochs. Part 3 curve: 64 / 16 / 3."""
    set_seed(SEED)                            # fresh, reproducible init per run
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=NUM_LABELS)
    args = TrainingArguments(
        output_dir=out_dir, seed=SEED,
        num_train_epochs=epochs, per_device_train_batch_size=batch,
        per_device_eval_batch_size=64, learning_rate=2e-5,
        eval_strategy='no', save_strategy='no',
        logging_strategy='epoch' if log_epochs else 'no',
        report_to='none', disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=encode(train_df, max_len))
    trainer.train()
    trainer.eval_max_len = max_len            # so eval_split tokenizes like training did
    return trainer


def eval_split(trainer, df, max_len=None):
    """Predict on df and score it. Defaults to the max_len the trainer was trained with."""
    max_len = max_len or getattr(trainer, 'eval_max_len', MAX_LEN)
    pred = trainer.predict(encode(df, max_len)).predictions.argmax(-1)
    return eu.evaluate(df['label'].values, pred)


def logged(method, full_row=False):
    """Latest TEST row for (MODEL, method) from the shared scoreboard. The eval module
    keys rows on (model, method, split, n_train_labeled) and no longer tracks person, so
    we match on MODEL + method here (each method has a single n in this notebook).
    Delete a row from results.csv to force that experiment to re-run."""
    if not eu.RESULTS_CSV.exists():
        return None
    r = pd.read_csv(eu.RESULTS_CSV)
    r = r[(r['model'] == MODEL) & (r['method'] == method) & (r['split'] == 'test')]
    if not len(r):
        return None
    row = r.iloc[-1]
    return row if full_row else {k: row[k] for k in eu.METRIC_KEYS}


def notes_kv(notes):
    """Parse the 'k=v; k=v' segments of a notes string into a dict (prose segments
    ignored). Used to validate cached rows against the current inputs/protocol."""
    out = {}
    for seg in str(notes).split(';'):
        if '=' in seg:
            k, v = seg.split('=', 1)
            out[k.strip()] = v.strip()
    return out


fmt = eu.fmt


def delta_vs(m, ref_method):
    ref = logged(ref_method)
    if ref is not None:
        print(f"delta macro-F1 (vs {ref_method}): "
              f"{float(m['f1_macro']) - float(ref['f1_macro']):+.4f}")


# --- 2a: 32-shot fine-tune (RESUME-AWARE: reuses the logged row; ~6 min on CPU otherwise) ---
shot_m = logged('32-shot')
if shot_m is None:
    trainer = train_bert(labeled_32, '../.cache/bert_32shot', log_epochs=True)
    val_m = eval_split(trainer, val)
    shot_m = eval_split(trainer, test)
    # headline numbers logged on TEST; val macro-F1 kept in notes for the tuning record
    eu.log_result(MODEL, '32-shot', len(labeled_32), shot_m,
                  notes=f"val_f1_macro={val_m['f1_macro']:.4f}")
    print('VAL :', fmt(val_m))
    print('[trained]')
else:
    print('[cached] (val macro-F1 is in the notes column of results.csv)')
print('2a 32-shot TEST:', fmt(shot_m))

### 2a - analysis

This is a question that allows to check how good is a BERT model adapted on only a few annotated examples compared with a fully trained model or a transparent baseline.‍ Full fine-tuning of `bert-base-uncased` on 32 labels (max_len 128, batch 8, 20 epochs, lr 2e-5) reaches 0.601 macro-F1 on the test split.‍ That's almost double the random floor of about 0.30, so the model is clearly learning, but it's still below the rule-based baseline of 0.73.‍ This means that in the extreme few-shot regime a 110M-parameter model fine-tuned on 32 examples can't compete with a simple lexicon: BERT could do that if it's properly adapted with enough data.‍

The per-class numbers show the model leaning on the majority.‍ Neutral F1 is strong at 0.82 while the minority classes lag, with negative F1 at only 0.465.‍ With about eleven negative examples there is not enough signal to shape a reliable decision boundary for that class, so the classifier defaults toward the dominant neutral region.‍ This is the same imbalance dynamic flagged in the Part-1 analysis, now reappearing inside a neural model.‍

The val/test gap signals instability rather than a real improvement: validation macro-F1 (0.508) is still below the test value (0.601), and with only 32 training examples a single-run estimate is noisy.‍

## Part 2b.‍ Train on Person D's augmented set (1)
Person D produces a non-LLM augmented training set (back-translation / EDA / etc.) under `data/augmented_32.csv`.‍ Re-train the SAME BERT on it and compare to 2a.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2b: re-train the SAME BERT (identical 2a protocol via train_bert) on Person D's
# back-translation augmented set; evaluate on test and compare to 2a.
# RESUME-AWARE: reuses the logged 'augmented' row (training is ~40 min on CPU).
# NOTE n_train_labeled=225 counts training ROWS; the real-label budget is still 32
# (the 225 rows are the 32 originals + paraphrases of them) — see the 2b analysis.
aug = pd.read_csv(AUG_CSV)
print('augmented set:', len(aug), 'rows ·', dict(aug['label_name'].value_counts()))

aug_m = logged('augmented')
if aug_m is None:
    aug_trainer = train_bert(aug, '../.cache/bert_2b')
    aug_val_m = eval_split(aug_trainer, val)
    aug_m = eval_split(aug_trainer, test)
    eu.log_result(MODEL, 'augmented', len(aug), aug_m,
                  notes=f"back-translation aug from 32; n={len(aug)}; val_f1_macro={aug_val_m['f1_macro']:.4f}")
    print('[trained]')
else:
    print('[cached]')

print('2b augmented TEST:', fmt(aug_m))
delta_vs(aug_m, '32-shot')

### 2b - analysis

Back-translation increases macro-F1 from 0.601 to 0.636 and accuracy from 0.68 to 0.74, applying the identical fine-tuning protocol to a set that combines the 32 originals with their paraphrases.‍ The augmentation idea is masking an over-represented token to force the model to use context; also, the 225 rows are paraphrases of the same 32 sentences, so they add no new information.‍ In this case, the gain reflects regularisation and decision-boundary smoothing rather than new signal, which is exactly why it is small.‍

This is confirmed with the per-class breakdown: The improvement shows up in neutral F1 (0.82 to 0.89) and positive F1 (0.515 to 0.549), while negative F1 is essentially flat (0.465 to 0.467).‍ Paraphrasing cannot manufacture the missing negative signal, so the minority-class problem survives augmentation untouched.‍

## Part 2d.‍ Train on 32 + Person D's LLM-generated data (1)
Person D produces `data/llm_generated.csv`.‍ Train BERT on the 32 + generated points; analyse impact on metrics.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2d: train the SAME BERT (identical 2a protocol via train_bert) on the 32 real labels
# + Person D's LLM-generated points; evaluate on test.
# RESUME-AWARE + STALENESS-SAFE: a logged 'llm-generated' row is reused only while it
# matches the current llm_generated.csv (row count recorded in notes as n=...). If
# Person D revises the file, this cell retrains automatically instead of serving a row
# trained on the obsolete data. Skips cleanly while the file has not been committed yet.
prev = logged('llm-generated', full_row=True)
gen_m = None

if os.path.exists(GEN_CSV):
    gen = pd.read_csv(GEN_CSV)
    combo_2d = pd.concat([labeled_32[['text', 'label']], gen[['text', 'label']]], ignore_index=True)
    if prev is not None and notes_kv(prev['notes']).get('n') == str(len(combo_2d)):
        gen_m = {k: prev[k] for k in eu.METRIC_KEYS}
        print('[cached]')
    else:
        if prev is not None:
            print('[stale] llm_generated.csv changed since the logged row — retraining')
        print(f'training set: 32 real + {len(gen)} generated = {len(combo_2d)} rows ·',
              dict(combo_2d['label'].value_counts().sort_index()))
        gen_trainer = train_bert(combo_2d, '../.cache/bert_2d')
        gen_val_m = eval_split(gen_trainer, val)
        gen_m = eval_split(gen_trainer, test)
        eu.log_result(MODEL, 'llm-generated', len(combo_2d), gen_m,
                      notes=f"n={len(combo_2d)}; gen={len(gen)}; val_f1_macro={gen_val_m['f1_macro']:.4f}")
        print('[trained]')
elif prev is not None:                # row exists but the file is gone — keep the row
    gen_m = {k: prev[k] for k in eu.METRIC_KEYS}
    print('[cached] (llm_generated.csv not present to validate against)')
else:
    print('[waiting] ../data/llm_generated.csv not committed yet (Person D).')

if gen_m is not None:
    print('2d 32+LLM-gen TEST:', fmt(gen_m))
    delta_vs(gen_m, '32-shot')
    delta_vs(gen_m, 'augmented')

We observe a significative improvement when we use LLM-generated data.‍ Training on the 32 real labels plus 360 LLM-generated sentences, checked for leakage against all three splits with none found, reaches 0.841 macro-F1 and 0.865 accuracy.‍ That represents a jump of about +0.24 over the 32-shot model and +0.21 over back-translation, and it is the first limited-label variant to clear the rule-based baseline of 0.73.‍

The mechanism is exactly what 2b is missing.‍ Where back-translation only paraphrased the existing 32 sentences and so added no new information, the LLM generated genuinely new labelled examples, including about 120 new negatives.‍ We can see that the per-class numbers make the point concrete: negative F1 rises from 0.465 in 2a and 0.467 in 2b to 0.874, also the LLM-generated data fixes the minority class that every earlier method failed on.‍ This strategy is far more effective than non-LLM augmentation.‍

The bottleneck in the limited-label regime is information, not model capacity: paraphrasing the same 32 sentences cannot help, whereas synthesising new labelled examples can.‍

`claude-opus-4-8` reaches 0.978 macro-F1 with zero labelled examples, and the few-shot variant with all 32 in-context examples logs byte-identical metrics.‍ This is the course's zero-shot-LLM-beats-the-fine-tuned-baseline thesis realised on this dataset, subject to the contamination and cost-and-latency caveats.‍

## Part 2 — comparison of the four model/technique combinations
The assignment compares four approaches on the shared `test` split: the 32-shot BERT (2a), the two augmentation strategies (2b back-translation, 2d LLM-generated), and the 2c zero-shot LLM.‍ The zero-shot and the optional few-shot runs are produced in `zeroshot_part2.ipynb`; here we only read their logged row.‍ The table is built live from `results/results.csv`, so the zero-shot row appears automatically once it has been logged, and is sorted by macro-F1.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2 comparison: the four model/technique combinations the assignment asks for —
# (2a) 32-shot BERT, (2b) +back-translation, (2d) +LLM-generated, and the (2c) zero-shot
# LLM logged in zeroshot_part2.ipynb. Read straight from the shared scoreboard so the
# table stays correct no matter who logged each row; a row not yet logged is reported as
# [waiting] rather than breaking the cell.
res = pd.read_csv(eu.RESULTS_CSV)
res = res[res['split'] == 'test']

combos = [
    ('2a 32-shot (BERT)',          res['model'].eq(MODEL) & res['method'].eq('32-shot')),
    ('2b back-translation (BERT)', res['model'].eq(MODEL) & res['method'].eq('augmented')),
    ('2d LLM-generated (BERT)',    res['model'].eq(MODEL) & res['method'].eq('llm-generated')),
    ('2c zero-shot (LLM)',         res['method'].eq('zero-shot')),
]
rows = []
for label, mask in combos:
    hit = res[mask]
    if len(hit):
        r = hit.iloc[-1]
        rows.append({'combination': label, 'model': r['model'],
                     'n_train_labeled': r['n_train_labeled'],
                     'accuracy': float(r['accuracy']),
                     'f1_macro': float(r['f1_macro']),
                     'f1_weighted': float(r['f1_weighted'])})
    else:
        print(f'[waiting] no logged row yet for: {label}')

comparison = (pd.DataFrame(rows)
              .sort_values('f1_macro', ascending=False)
              .reset_index(drop=True))
comparison.round(4)

## Part 2e.‍ Optimal technique (0.5)
Apply the best technique(s) from 2a/2b/2d.‍ Comment + propose improvements (student-authored).‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2e: apply the most effective technique(s) from 2a/2b/2d — combine ALL training
# sources (32 real + augmented + LLM-generated), dedup on text, train with the SAME 2a
# protocol so the row stays comparable.
# Requires the 2d data: augmented_32.csv already CONTAINS the 32 originals, so a
# "32 + augmented" combo would dedup back to exactly the 2b training set — training it
# again would waste ~45 CPU-minutes to reproduce the 2b row.
# RESUME-AWARE + STALENESS-SAFE: the logged 'optimal-combo' row is reused only if both
# the recipe AND the deduped row count (recorded as k=v in notes) match the current
# input files; if either source CSV changes, it retrains automatically.
part2 = {m: logged(m) for m in ['32-shot', 'augmented', 'llm-generated']}
part2 = {m: v for m, v in part2.items() if v is not None}
if part2:
    board = (pd.DataFrame(part2).T[['accuracy', 'f1_macro']].astype(float)
             .sort_values('f1_macro', ascending=False))
    print('Part 2 scoreboard (test):')
    print(board.round(4), '\n')

if not os.path.exists(GEN_CSV):
    print('[waiting] 2e needs ../data/llm_generated.csv (see note above) — re-run once 2d data lands.')
else:
    recipe = '32-real+augmented+llm-generated'
    combo_2e = (pd.concat([labeled_32[['text', 'label']],
                           pd.read_csv(AUG_CSV)[['text', 'label']],
                           pd.read_csv(GEN_CSV)[['text', 'label']]], ignore_index=True)
                .drop_duplicates(subset='text', ignore_index=True))
    prev = logged('optimal-combo', full_row=True)
    kv = notes_kv(prev['notes']) if prev is not None else {}
    if kv.get('recipe') == recipe and kv.get('n') == str(len(combo_2e)):
        best_m = {k: prev[k] for k in eu.METRIC_KEYS}
        print(f'[cached] optimal-combo ({recipe}, n={len(combo_2e)})')
    else:
        if prev is not None:
            print('[stale] input files changed since the logged row — retraining')
        print(f'optimal combo = {recipe} -> {len(combo_2e)} unique rows ·',
              dict(combo_2e['label'].value_counts().sort_index()))
        best_trainer = train_bert(combo_2e, '../.cache/bert_2e')
        best_val_m = eval_split(best_trainer, val)
        best_m = eval_split(best_trainer, test)
        eu.log_result(MODEL, 'optimal-combo', len(combo_2e), best_m,
                      notes=f"recipe={recipe}; n={len(combo_2e)}; val_f1_macro={best_val_m['f1_macro']:.4f}")
        print('[trained]')
    print('2e optimal-combo TEST:', fmt(best_m))
    if part2:
        top = board['f1_macro'].idxmax()
        print(f"delta macro-F1 (optimal-combo - best single = {top}): "
              f"{float(best_m['f1_macro']) - float(board.loc[top, 'f1_macro']):+.4f}")

### 2e - comments & proposed improvements

Combining all sources does not beat the best single one: the optimal experiment pools the 32 real labels, the back-translation set, and the LLM-generated set, deduplicated to 585 rows, and trains with the same protocol.‍ It reaches 0.800 macro-F1, comfortably above the rule-based baseline but below 2d's 0.841.‍

Adding the back-translation paraphrases to the clean LLM-generated data actually hurt, and the damage falls exactly where it should: negative F1 drops from 0.874 in 2d to 0.774.‍ 

The lesson is that more data is not automatically better.‍ The low-quality, no-new-information paraphrases introduced noise that diluted the genuine signal of the generated examples, so the best Part-2 technique is 2d (32 plus LLM-generated) on its own.‍ So, curation beats blind concatenation.‍